# CSV to Anki (.apkg) converter

1. Run all cells (*Runtime → Run all*).
2. When the text box appears, paste your CSV and click **Create deck**. **Column 1 = question, column 2 = answer.** Extra columns are ignored.
3. The generated `.apkg` file is downloaded automatically. Open it with Anki (desktop: double-click / *File → Import*; mobile: share/open with AnkiDroid or AnkiMobile).

In [ ]:
!pip install -q genanki

In [ ]:
#@title Settings
DECK_NAME = "My CSV Deck"  #@param {type:"string"}
HAS_HEADER_ROW = False  #@param {type:"boolean"}
ESCAPE_HTML = True  #@param {type:"boolean"}

In [ ]:
#@title Paste CSV
from IPython.display import Javascript, display
from google.colab.output import eval_js

PASTE_JS = """
async function pasteCsv() {
  const box = document.createElement('div');
  const ta = document.createElement('textarea');
  ta.rows = 15;
  ta.style.width = '100%';
  ta.placeholder = 'Paste CSV here (column 1 = question, column 2 = answer)';
  const btn = document.createElement('button');
  btn.textContent = 'Create deck';
  btn.style.marginTop = '8px';
  box.append(ta, document.createElement('br'), btn);
  document.body.appendChild(box);
  google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  ta.focus();
  await new Promise(resolve => {
    btn.onclick = () => ta.value.trim() ? resolve() : alert('Please paste some CSV text first.');
  });
  const value = ta.value;
  box.remove();
  return value;
}
"""

display(Javascript(PASTE_JS))
# Blocks until the button is clicked, so "Run all" works.
text = eval_js("pasteCsv()").strip("\r\n")
print(f"Received {len(text.splitlines())} lines.")

In [ ]:
import csv
import html
import io

try:
    dialect = csv.Sniffer().sniff(text[:4096], delimiters=",;\t|")
except csv.Error:
    dialect = csv.excel

rows = list(csv.reader(io.StringIO(text), dialect))
if HAS_HEADER_ROW:
    rows = rows[1:]

def to_field(value: str) -> str:
    value = value.strip()
    if ESCAPE_HTML:
        value = html.escape(value)
    return value.replace("\r\n", "\n").replace("\n", "<br>")

cards = []
skipped = 0
for row in rows:
    if len(row) < 2 or not row[0].strip() or not row[1].strip():
        skipped += 1
        continue
    cards.append((to_field(row[0]), to_field(row[1])))

print(f"Delimiter: {dialect.delimiter!r} | cards: {len(cards)} | skipped rows: {skipped}")
for q, a in cards[:5]:
    print(f"Q: {q}\nA: {a}\n")

In [ ]:
import hashlib
import re
import genanki
from google.colab import files

if not cards:
    raise SystemExit("No valid rows found in the CSV.")

def stable_id(s: str) -> int:
    return int(hashlib.sha1(s.encode()).hexdigest()[:8], 16) | (1 << 30)

model = genanki.Model(
    stable_id("csv-to-anki-basic-model"),
    "CSV Basic (Question/Answer)",
    fields=[{"name": "Question"}, {"name": "Answer"}],
    templates=[{
        "name": "Card 1",
        "qfmt": "{{Question}}",
        "afmt": "{{FrontSide}}<hr id=\"answer\">{{Answer}}",
    }],
    css=".card { font-family: arial; font-size: 20px; text-align: center; color: black; background-color: white; }",
)

deck = genanki.Deck(stable_id(DECK_NAME), DECK_NAME)
for q, a in cards:
    deck.add_note(genanki.Note(model=model, fields=[q, a], guid=genanki.guid_for(DECK_NAME, q, a)))

safe_name = re.sub(r"[^\w\-]+", "_", DECK_NAME).strip("_") or "deck"
out_file = f"{safe_name}.apkg"
genanki.Package(deck).write_to_file(out_file)
print(f"Created {out_file} with {len(cards)} cards.")

files.download(out_file)